# Limpieza (IV-C) + extracción de skills con NLP (IV-E)
Entrada: `empleos_computrabajo.xlsx` (del notebook de scraping).

In [1]:
#%pip install -q spacy scikit-learn openpyxl
#!python -m spacy download es_core_news_sm -q

In [2]:
import re, html
import pandas as pd, numpy as np
from bs4 import BeautifulSoup
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

df = pd.read_excel("empleos_computrabajo.xlsx")
print("Inicial:", len(df))

def limpiar(t):
    if not isinstance(t, str): return ""
    t = BeautifulSoup(html.unescape(t), "html.parser").get_text(" ")
    t = re.sub(r"[^\w\s.,;:()/+#%$\-S]", " ", t)   # caracteres especiales
    return re.sub(r"\s+", " ", t).strip()

for c in ["titulo", "descripcion", "requisitos"]:
    df[c] = df[c].map(limpiar)

# Duplicados: exacto por URL y aproximado por coseno entre descripciones
df = df.drop_duplicates("url").reset_index(drop=True)
X = TfidfVectorizer().fit_transform(df["descripcion"])
dup, UMBRAL = set(), 0.95
for i in range(0, X.shape[0], 500):
    S = cosine_similarity(X[i:i+500], X)
    for r, fila in enumerate(S):
        j = i + r
        if j in dup: continue
        dup.update(k for k in np.where(fila >= UMBRAL)[0] if k > j)
df = df.drop(index=list(dup)).reset_index(drop=True)
print("Tras deduplicar:", len(df))

Inicial: 740


C:\Users\Rodrigo Fernández\AppData\Local\Temp\ipykernel_32232\642701459.py:12: MarkupResemblesLocatorWarning: The input looks more like a filename than markup. You may want to open this file and pass the filehandle into Beautiful Soup.
  t = BeautifulSoup(html.unescape(t), "html.parser").get_text(" ")


Tras deduplicar: 686


In [3]:
# Seniority desde el título (palabras clave)
def seniority(t):
    t = t.lower()
    for nivel, pat in [("practicante", r"practicante|trainee|intern\b"),
                       ("semi senior", r"semi\s?-?senior|\bssr\b"),
                       ("senior", r"senior|\bsr\b"),
                       ("junior", r"junior|\bjr\b"),
                       ("lead", r"lead|l[ií]der|jefe|gerente|arquitect|manager")]:
        if re.search(pat, t): return nivel
    return "no especificado"
df["seniority"] = df["titulo"].map(seniority)

# Salario -> soles mensuales (min, max)
def salario(s):
    if not isinstance(s, str): return (np.nan, np.nan)
    s = re.sub(r"(?<=\d)[.,](?=\d{3}\b)", "", s)
    n = [float(x) for x in re.findall(r"\d+(?:[.,]\d+)?", s.replace(",", "."))]
    if not n: return (np.nan, np.nan)
    return (min(n), max(n))
df[["sal_min", "sal_max"]] = df["salario"].apply(lambda s: pd.Series(salario(s)))
df["sal_mid"] = df[["sal_min", "sal_max"]].mean(axis=1)
print("Con salario declarado: %.1f%%" % (100 * df["sal_min"].notna().mean()))

Con salario declarado: 27.8%


In [4]:
import spacy
from spacy.matcher import PhraseMatcher

# Catálogo canónico -> alias (ampliar / mapear a ESCO)
CATALOGO = {
 "python": ["python"], "java": ["java"], "javascript": ["javascript", "js"],
 "typescript": ["typescript"], "c#": ["c#", "c sharp"], "c++": ["c++"],
 "php": ["php"], "golang": ["golang"], "kotlin": ["kotlin"], "swift": ["swift"],
 "r": ["lenguaje r", "rstudio"], "sql": ["sql", "t-sql", "plsql", "pl/sql"],
 "nosql": ["nosql"], "mongodb": ["mongodb", "mongo db"], "postgresql": ["postgresql", "postgres"],
 "mysql": ["mysql"], "sql server": ["sql server", "sqlserver"], "oracle": ["oracle"],
 "power bi": ["power bi", "powerbi"], "tableau": ["tableau"], "excel": ["excel"],
 "pandas": ["pandas"], "numpy": ["numpy"], "scikit-learn": ["scikit-learn", "sklearn", "scikit learn"],
 "tensorflow": ["tensorflow"], "pytorch": ["pytorch"], "machine learning": ["machine learning", "aprendizaje automático", "aprendizaje automatico"],
 "deep learning": ["deep learning", "aprendizaje profundo"], "nlp": ["nlp", "procesamiento de lenguaje natural"],
 "estadística": ["estadística", "estadistica"], "etl": ["etl", "elt"],
 "spark": ["spark", "pyspark"], "hadoop": ["hadoop"], "kafka": ["kafka"], "airflow": ["airflow"],
 "dbt": ["dbt"], "databricks": ["databricks"], "data warehouse": ["data warehouse", "datawarehouse", "dwh"],
 "big data": ["big data"], "aws": ["aws", "amazon web services"], "azure": ["azure"],
 "gcp": ["gcp", "google cloud"], "docker": ["docker"], "kubernetes": ["kubernetes", "k8s"],
 "terraform": ["terraform"], "jenkins": ["jenkins"], "git": ["git", "github", "gitlab"],
 "ci/cd": ["ci/cd", "cicd", "integración continua", "integracion continua"], "linux": ["linux"],
 "react": ["react", "reactjs", "react.js"], "angular": ["angular"], "vue": ["vue", "vue.js", "vuejs"],
 "node.js": ["node.js", "nodejs", "node js"], "spring": ["spring", "spring boot"], ".net": [".net", "dotnet", "asp.net"],
 "django": ["django"], "flask": ["flask"], "fastapi": ["fastapi"], "html/css": ["html", "css", "html5", "css3"],
 "api rest": ["api rest", "rest api", "restful", "apis"], "microservicios": ["microservicios", "microservicio", "microservices"],
 "scrum": ["scrum"], "agile": ["agile", "ágil", "agil", "metodologías ágiles", "metodologias agiles"],
 "jira": ["jira"], "ciberseguridad": ["ciberseguridad", "seguridad informática", "seguridad informatica", "cybersecurity"],
 "pentesting": ["pentesting", "pentest", "hacking ético"], "siem": ["siem", "splunk"], "iso 27001": ["iso 27001", "iso27001"],
 "redes": ["redes", "cisco", "ccna"], "ingles": ["inglés", "ingles", "english"],
 "comunicación": ["comunicación", "comunicacion"], "trabajo en equipo": ["trabajo en equipo"],
 "liderazgo": ["liderazgo"], "testing": ["testing", "pruebas unitarias", "qa", "selenium", "pytest"],
 "devops": ["devops"], "mlops": ["mlops"], "power automate": ["power automate"], "sap": ["sap"],
 "autocad": ["autocad", "auto cad"], "ms project": ["ms project", "microsoft project"],
 "solidworks": ["solidworks"], "revit/bim": ["revit", "bim"], "office": ["office", "ms office", "microsoft office"],
 "vba": ["vba", "macros"], "power apps": ["power apps", "powerapps"], "erp": ["erp"], "kpi": ["kpi", "indicadores"],
 "ssoma": ["ssoma", "sst", "seguridad y salud ocupacional"], "iso 9001": ["iso 9001"], "iso 45001": ["iso 45001"],
 "iso 14001": ["iso 14001"], "mantenimiento": ["mantenimiento", "mantenimiento preventivo", "mantenimiento correctivo"],
 "gestión de proyectos": ["gestión de proyectos", "gestion de proyectos", "pmbok", "pmp"],
 "costos y presupuestos": ["costos", "presupuestos", "s10", "opus"], "planeamiento": ["planeamiento", "planificación"],
 "licencia de conducir": ["licencia de conducir", "licencia de conducir a1", "licencia de conducir b"],
}

nlp = spacy.load("es_core_news_sm", disable=["parser", "ner"])
matcher = PhraseMatcher(nlp.vocab, attr="LEMMA")
for skill, alias in CATALOGO.items():
    matcher.add(skill, list(nlp.pipe(a.lower() for a in alias)))

def extraer_skills(textos):
    out = []
    for doc in nlp.pipe(t.lower() for t in textos):   # tokenización + lematización
        out.append(sorted({nlp.vocab.strings[m] for m, s, e in matcher(doc)}))
    return out

df["texto_skills"] = df["requisitos"].replace("No especificado", "") + ". " + df["descripcion"].replace("No especificado", "")
df["skills"] = extraer_skills(df["texto_skills"])
print("Promedio skills/oferta:", df["skills"].map(len).mean())
print(df["skills"].explode().value_counts().head(20))

Promedio skills/oferta: 3.3556851311953353
skills
excel                    221
mantenimiento            215
comunicación             143
autocad                  141
kpi                      128
costos y presupuestos    122
planeamiento             113
ingles                   109
trabajo en equipo        103
ssoma                     96
office                    93
liderazgo                 76
licencia de conducir      72
power bi                  70
redes                     69
gestión de proyectos      57
sap                       40
revit/bim                 38
ms project                38
iso 9001                  35
Name: count, dtype: int64


In [5]:
# Salidas
df.drop(columns="texto_skills").assign(skills=df["skills"].map("; ".join)).to_excel("ofertas_limpias_skills.xlsx", index=False)
df[["url", "seniority", "sal_mid", "skills"]].explode("skills").dropna(subset=["skills"]).to_csv("oferta_skill.csv", index=False)

## Validación del extractor (IV-E)
Referencia = búsqueda de cadenas con regex sobre el texto crudo (sin spaCy), usando los mismos alias. Mide la consistencia del matching por lemas frente a una búsqueda literal; no detecta skills fuera del catálogo.

In [6]:

import unicodedata
quitar = lambda s: unicodedata.normalize("NFKD", s.lower()).encode("ascii", "ignore").decode()
pats = {k: [re.compile(r"(?<!\w)" + re.escape(quitar(a)) + r"(?!\w)") for a in al] for k, al in CATALOGO.items()}

ref = [{k for k, ps in pats.items() if any(p.search(txt) for p in ps)} for txt in map(quitar, df.texto_skills)]
sis = [set(s) for s in df.skills]

tp = sum(len(s & g) for s, g in zip(sis, ref)); fp = sum(len(s - g) for s, g in zip(sis, ref)); fn = sum(len(g - s) for s, g in zip(sis, ref))
P, R = tp / (tp + fp), tp / (tp + fn)
print(f"Precisión={P:.3f}  Exhaustividad={R:.3f}  F1={2*P*R/(P+R):.3f}  (n={len(df)} ofertas)")

from collections import Counter
print("Solo spaCy (posibles falsos positivos):", Counter(x for s, g in zip(sis, ref) for x in s - g).most_common(8))
print("Solo regex (omitidas por spaCy):", Counter(x for s, g in zip(sis, ref) for x in g - s).most_common(8))


Precisión=0.959  Exhaustividad=0.959  F1=0.959  (n=686 ofertas)
Solo spaCy (posibles falsos positivos): [('licencia de conducir', 33), ('redes', 18), ('costos y presupuestos', 14), ('comunicación', 12), ('estadística', 5), ('mantenimiento', 5), ('kpi', 4), ('vba', 1)]
Solo regex (omitidas por spaCy): [('office', 39), ('ssoma', 20), ('testing', 13), ('jira', 4), ('iso 45001', 4), ('azure', 3), ('power automate', 2), ('iso 9001', 2)]
